# 冬休みの課題：囚人のジレンマ戦略大会

第10回・第11回では，スロットマシンという「確率は分からないが，こちらの行動に反応しない」相手に対する戦略を考えました。今回の相手は，**こちらの行動を見て，出方を変えてくる相手**です。

各自が戦略を1つ作って提出し，1月の最初の授業で，全員の戦略による大会を行います。

## 囚人のジレンマ

2人のプレーヤーが，それぞれ「協調（C）」か「裏切り（D）」かを同時に選びます。得点は次の通りです（左が自分，右が相手の得点）。

|          | 相手 C | 相手 D |
|----------|--------|--------|
| **自分 C** | 3, 3   | 0, 5   |
| **自分 D** | 5, 0   | 1, 1   |

相手が C でも D でも，自分は D を選んだ方が得点が高くなります。ところが，2人とも D を選ぶと 1 点ずつで，2人とも C を選んだ時の 3 点ずつより悪くなってしまいます。これが「ジレンマ」です。

**考えよう**：1回だけの勝負なら，D を選ぶのが合理的に見えます。では，同じ相手と何度もくり返して対戦する場合はどうでしょうか。

## くり返し囚人のジレンマ

同じ相手と何度も対戦し，合計得点を競います。1980年ごろ，政治学者の Axelrod が，研究者から戦略のプログラムを募集して大会を開いたところ，優勝したのは「しっぺ返し（Tit for Tat）」という，たった数行のプログラムでした。最初は協調し，あとは相手の前の手をまねるだけです。しっぺ返しは，どの相手にも「勝つ」ことはありません（相手より得点が高くなることはない）が，合計得点では一番になりました。

この課題の大会は，次のルールで行います。

* 1回の対戦は，毎回 1% の確率で終わります（平均 100 回）。何回で終わるかは分かりません。
* 2% の確率で，出した手が反転します（手が滑る，ノイズ）。
* 得点は，1回あたりの平均得点（0 から 5 の間）で表します。

これらの値は，pd.py の `PEND`, `NOISE`, `PAYOFF` に書かれています。

## 戦略の作り方

pd.py を import して使います。bandit.py と同じように，このファイルと同じフォルダに pd.py を置いてください。

In [1]:
from pd import *

戦略は，次の2つのメソッドを持つクラスとして作ります。第10回の多腕バンディットの戦略とほとんど同じ形です。

`choice(self)`: 次の手 'C' か 'D' を返す。

`learn(self, my, other)`: 今の回の自分の手 my と相手の手 other を教えてもらう。ノイズがある時は，反転した後の，実際に出た手が教えられます。

1回の対戦ごとに新しいインスタンスが作られるので，reset は要りません。必要なら `__init__(self)` で初期化します。

例えば，しっぺ返しは次のように書けます（pd.py の中の TitForTat と同じものです）。

In [2]:
class MyTitForTat:
    def __init__(self):
        self.my = 'C'     # 次に出す手

    def choice(self):
        return self.my

    def learn(self, my, other):
        self.my = other   # 相手の手をまねる

`Match(A, B)` で，戦略 A と B を1回対戦させることができます。(A の平均得点, B の平均得点, 回数) が返ってきます。

In [3]:
print(Match(MyTitForTat, AllD))
print(Match(MyTitForTat, AllC))
print(Match(MyTitForTat, MyTitForTat))

(0.9166666666666666, 1.3333333333333333, 12)
(3.0197368421052633, 2.9046052631578947, 304)
(3.0, 3.0, 16)


## 主催者の戦略

pd.py には，次の 11 個の戦略が用意されています（リスト `HOUSE`）。中身は pd.py を読んでください。

| クラス名 | 内容 |
|---|---|
| AllC | いつも協調 |
| AllD | いつも裏切り |
| RandomS | ランダム |
| Alternate | 協調と裏切りを交互に |
| TitForTat | しっぺ返し |
| SuspiciousTFT | 最初は裏切るしっぺ返し |
| GenerousTFT | 相手の裏切りを 1/3 の確率で許すしっぺ返し |
| TitForTwoTats | 相手が2回続けて裏切ったら裏切る |
| Grim | 一度でも裏切られたら，ずっと裏切る |
| Pavlov | 得点が 3 か 5 なら同じ手，0 か 1 なら手を変える |
| Joss | しっぺ返しだが，10% の確率でこっそり裏切る |

**考えよう**：Pavlov は，第10回のどの戦略と同じ考え方でしょうか。

## 採点

`Evaluate(S)` は，戦略 S を主催者の 11 個の戦略とそれぞれ 100 回ずつ対戦させ，相手ごとの平均得点と，全体の平均得点，そして**対戦点**（10 点満点）を表示します。乱数の種を固定しているので，同じプログラムなら何度実行しても同じ結果になります。

In [4]:
res, total = Evaluate(MyTitForTat)

          AllC : 3.013
          AllD : 0.985
       RandomS : 2.217
     Alternate : 2.471
     TitForTat : 2.591
 SuspiciousTFT : 2.271
   GenerousTFT : 2.952
 TitForTwoTats : 3.014
          Grim : 1.913
        Pavlov : 2.519
          Joss : 1.931
           平均 : 2.352
         対戦点 : 6.0 / 10


対戦点は，しっぺ返しと同じ平均得点で 6 点，それより 0.05 高いごとに 1 点増え（低ければ減り），0 点から 10 点の範囲におさめたものです。提出されたプログラムに `Evaluate` を実行した結果が，そのまま成績の対戦点になります。**提出前に自分で何度でも確かめて，改良してください。**

主催者の戦略の中にも，しっぺ返しより点の高いものがあります。

In [5]:
for S in HOUSE:
    res, total = Evaluate(S, show=False)
    print(f'{S.__name__:>14s} : {total:.3f}  対戦点 {Points(total)}')

          AllC : 2.193  対戦点 2.8


          AllD : 2.163  対戦点 2.2
       RandomS : 2.265  対戦点 4.3


     Alternate : 2.360  対戦点 6.2


     TitForTat : 2.352  対戦点 6.0
 SuspiciousTFT : 2.213  対戦点 3.2


   GenerousTFT : 2.394  対戦点 6.8


 TitForTwoTats : 2.368  対戦点 6.3


          Grim : 2.268  対戦点 4.3
        Pavlov : 2.351  対戦点 6.0


          Joss : 2.232  対戦点 3.6


## 課題

自分の戦略を作り，`自分の名字 + "PD"` という名前のクラスにして，`自分の名字 + "PD.py"` というファイルに保存しよう（例：TsuikiPD クラスを TsuikiPD.py に）。第10回と同じく，import して正しく動くことを確かめてから提出すること。

あわせて，次のことを書いたレポート（ipynb ファイル）を提出しよう。

1. **戦略の説明**：どういう考えで作ったか。どの相手にどう対応するか。
2. **予想**：`Evaluate` を実行する前に，自分の戦略はどの相手に強く，どの相手に弱いと予想したか。学生どうしの大会では何位くらいになると思うか。
3. **結果と考察**：予想と結果のずれと，その理由。改良の過程で何を試し，何が効いたか（効かなかったか）。
4. **AI の利用**：AI を使ったなら，何に使ったか。

配点は，対戦点 10 点とレポート 5 点の，合計 15 点の予定です。

## 大会

提出された全員の戦略と主催者の戦略で，総当たり戦（自分自身のコピーとの対戦も含む）を行い，1月の最初の授業で結果を発表します。上位の人には，授業中に戦略の解説をしてもらいます。**大会の順位は成績には関係しません。**

総当たり戦は `Tournament` で行えます。主催者の戦略どうしで試してみよう。

In [6]:
M, total = Tournament(HOUSE)
ShowTournament(HOUSE, M, total)

   GenerousTFT : 2.384
     Alternate : 2.381
 TitForTwoTats : 2.332
        Pavlov : 2.327
          Grim : 2.321
     TitForTat : 2.317
 SuspiciousTFT : 2.258
          Joss : 2.244
       RandomS : 2.237
          AllC : 2.205
          AllD : 2.150


**考えよう**：採点（主催者の戦略との対戦）と学生どうしの大会では，有利な戦略が違うかもしれません。主催者の戦略に特化しすぎた戦略は，大会ではどうなるでしょうか。

## ルールと AI の使い方

次のことは禁止です。

* 相手のオブジェクトの中身を覗いたり，書き換えたりすること。pd.py の変数（NOISE など）を書き換えること。
* 乱数の種（`random.seed`）を操作すること。
* 他の参加者と示し合わせて戦略を作ること。
* 1回の `choice`, `learn` に長い時間をかけること（`Evaluate` が1分以内で終わるようにしよう）。

**この課題での AI の使い方**：第10回と同じく，問題そのものを AI に与えて戦略を作ってもらうことはしないでください。どういう戦略にするかは自分で考えよう。Python の書き方を聞いたり，エラーの意味を説明してもらったり，自分のアイデアをプログラムにする手伝いをしてもらうのは構いません。その場合は，レポートに書いてください。

提出されたプログラムについて，1月の授業で質問することがあります。自分のプログラムは，自分で説明できるようにしておこう。

## 考えるヒント

* ノイズがあると，しっぺ返しどうしはどうなるでしょうか。`Match(TitForTat, TitForTat, noise=0)` と `noise=0.02` を比べてみよう。
* 相手の手の履歴をリストにとっておけば，相手がどういう戦略かを推測することができます。第11回のベイズ推論の考え方も使えるかもしれません。
* 相手の戦略が決まっている時，乱数を使った戦略は，乱数を使わない戦略より有利になることがあるでしょうか。自分の戦略で乱数を使ったなら，乱数を使わないようにしたらどうなるか確かめてみよう。